In [ ]:
import random
import math


# ---------------------------------------------------------
# CLASE NODO
# ---------------------------------------------------------
class Nodo:
    """
    Representa un vértice dentro del grafo.

    Cada nodo tiene:
    - id : identificador único
    - x,y : coordenadas (usadas en el modelo geográfico)
    - aristas : lista de aristas que salen de este nodo
    """

    def __init__(self, id, x=None, y=None):
        """
        Constructor de la clase Nodo

        :param id: identificador único del nodo
        :param x: coordenada x (opcional)
        :param y: coordenada y (opcional)
        """
        self.id = id
        self.x = x
        self.y = y

        # Lista de aristas incidentes al nodo
        self.aristas = []

    def grado(self):
        """
        Calcula el grado del nodo.
        El grado es el número de aristas conectadas al nodo.

        :return: número de aristas
        """
        return len(self.aristas)

    def __repr__(self):
        """
        Representación en texto del nodo
        """
        return f"Nodo({self.id})"


# ---------------------------------------------------------
# CLASE ARISTA
# ---------------------------------------------------------
class Arista:
    """
    Representa una conexión entre dos nodos del grafo.
    """

    def __init__(self, origen, destino):
        """
        Constructor de la arista.

        :param origen: nodo desde donde sale la arista
        :param destino: nodo al que llega la arista
        """
        self.origen = origen
        self.destino = destino

    def __repr__(self):
        """
        Representación en texto de la arista
        """
        return f"{self.origen.id} -> {self.destino.id}"


# ---------------------------------------------------------
# CLASE GRAFO
# ---------------------------------------------------------
class Grafo:
    """
    Representa un grafo completo.

    Contiene:
    - nodos: diccionario de nodos
    - aristas: lista de aristas
    - dirigido: indica si el grafo es dirigido
    """

    def __init__(self, dirigido=False):
        """
        Constructor del grafo

        :param dirigido: indica si el grafo es dirigido
        """
        self.nodos = {}     # diccionario id -> Nodo
        self.aristas = []   # lista de aristas
        self.dirigido = dirigido

    # -----------------------------------------------------
    # Agregar nodo
    # -----------------------------------------------------
    def agregar_nodo(self, id, x=None, y=None):
        """
        Agrega un nodo al grafo.

        Si el nodo ya existe no se vuelve a crear.

        :param id: identificador del nodo
        :param x: coordenada x (opcional)
        :param y: coordenada y (opcional)
        """
        if id not in self.nodos:
            self.nodos[id] = Nodo(id, x, y)

    # -----------------------------------------------------
    # Agregar arista
    # -----------------------------------------------------
    def agregar_arista(self, origen, destino):
        """
        Agrega una arista entre dos nodos.

        Si el grafo no es dirigido se agregan dos aristas
        (una en cada dirección).

        :param origen: id del nodo origen
        :param destino: id del nodo destino
        """

        # evitar loops
        if origen == destino:
            return

        n1 = self.nodos[origen]
        n2 = self.nodos[destino]

        arista = Arista(n1, n2)

        # guardar arista en lista global
        self.aristas.append(arista)

        # guardar arista en nodo origen
        n1.aristas.append(arista)

        # si el grafo no es dirigido se agrega la inversa
        if not self.dirigido:
            arista2 = Arista(n2, n1)
            self.aristas.append(arista2)
            n2.aristas.append(arista2)

    # -----------------------------------------------------
    # Guardar grafo en formato GraphViz
    # -----------------------------------------------------
    def guardar_graphviz(self, archivo):
        """
        Exporta el grafo en formato .dot para GraphViz.

        GraphViz permite visualizar grafos fácilmente.

        :param archivo: nombre del archivo .dot
        """

        with open(archivo, "w") as f:

            # tipo de grafo
            tipo = "digraph" if self.dirigido else "graph"

            # símbolo de conexión
            conector = "->" if self.dirigido else "--"

            f.write(f"{tipo} G {{\n")

            # escribir nodos
            for nodo in self.nodos.values():
                f.write(f"  {nodo.id};\n")

            # escribir aristas
            for arista in self.aristas:

                # evitar duplicados en grafo no dirigido
                if not self.dirigido and arista.origen.id > arista.destino.id:
                    continue

                f.write(
                    f"  {arista.origen.id} {conector} {arista.destino.id};\n"
                )

            f.write("}\n")


# =========================================================
# MODELOS DE GENERACIÓN DE GRAFOS
# =========================================================


# ---------------------------------------------------------
# MODELO MALLA (GRID)
# ---------------------------------------------------------
def grafoMalla(m, n, dirigido=False):
    """
    Genera un grafo tipo malla.

    Cada nodo se conecta con:
    - el nodo a la derecha
    - el nodo de abajo

    :param m: columnas
    :param n: filas
    :param dirigido: si el grafo es dirigido
    """

    g = Grafo(dirigido)

    # crear nodos
    for i in range(m):
        for j in range(n):
            g.agregar_nodo((i, j))

    # crear conexiones
    for i in range(m):
        for j in range(n):

            # conexión vertical
            if i < m - 1:
                g.agregar_arista((i, j), (i + 1, j))

            # conexión horizontal
            if j < n - 1:
                g.agregar_arista((i, j), (i, j + 1))

    return g


# ---------------------------------------------------------
# MODELO ERDOS - RENYI
# ---------------------------------------------------------
def grafoErdosRenyi(n, m, dirigido=False):
    """
    Genera un grafo aleatorio seleccionando m aristas.

    :param n: número de nodos
    :param m: número de aristas
    """

    g = Grafo(dirigido)

    for i in range(n):
        g.agregar_nodo(i)

    # lista de todas las posibles aristas
    posibles = [(i, j) for i in range(n) for j in range(n) if i != j]

    # elegir m al azar
    aristas = random.sample(posibles, m)

    for (i, j) in aristas:
        g.agregar_arista(i, j)

    return g


# ---------------------------------------------------------
# MODELO GILBERT
# ---------------------------------------------------------
def grafoGilbert(n, p, dirigido=False):
    """
    Entre cada par de nodos existe una probabilidad p
    de que se cree una arista.

    :param n: número de nodos
    :param p: probabilidad de conexión
    """

    g = Grafo(dirigido)

    for i in range(n):
        g.agregar_nodo(i)

    for i in range(n):
        for j in range(n):

            if i == j:
                continue

            if not dirigido and j <= i:
                continue

            # decisión probabilística
            if random.random() <= p:
                g.agregar_arista(i, j)

    return g


# ---------------------------------------------------------
# MODELO GEOGRAFICO
# ---------------------------------------------------------
def grafoGeografico(n, r, dirigido=False):
    """
    Los nodos se colocan en un plano.
    Se conectan si la distancia entre ellos es <= r.

    :param n: número de nodos
    :param r: radio máximo de conexión
    """

    g = Grafo(dirigido)

    # generar posiciones aleatorias
    for i in range(n):
        x = random.random()
        y = random.random()
        g.agregar_nodo(i, x, y)

    nodos = list(g.nodos.values())

    # comparar distancias
    for i in range(n):
        for j in range(i + 1, n):

            dx = nodos[i].x - nodos[j].x
            dy = nodos[i].y - nodos[j].y

            dist = math.sqrt(dx**2 + dy**2)

            if dist <= r:
                g.agregar_arista(nodos[i].id, nodos[j].id)

    return g


# ---------------------------------------------------------
# MODELO BARABASI - ALBERT
# ---------------------------------------------------------
def grafoBarabasiAlbert(n, d, dirigido=False):
    """
    Modelo de crecimiento preferencial.

    Los nodos con mayor grado tienen mayor probabilidad
    de recibir nuevas conexiones.

    :param n: número de nodos
    :param d: número de conexiones del nodo nuevo
    """

    g = Grafo(dirigido)

    # crear núcleo inicial completamente conectado
    for i in range(d):
        g.agregar_nodo(i)

    for i in range(d):
        for j in range(i + 1, d):
            g.agregar_arista(i, j)

    # agregar nodos uno por uno
    for nuevo in range(d, n):

        g.agregar_nodo(nuevo)

        nodos_existentes = list(g.nodos.values())

        grados = [nodo.grado() for nodo in nodos_existentes]
        suma_grados = sum(grados)

        conectados = set()

        while len(conectados) < d:

            r = random.uniform(0, suma_grados)

            acumulado = 0

            for nodo in nodos_existentes:

                acumulado += nodo.grado()

                if acumulado >= r:
                    conectados.add(nodo.id)
                    break

        for v in conectados:
            g.agregar_arista(nuevo, v)

    return g


# ---------------------------------------------------------
# MODELO DOROGOVTSEV - MENDES
# ---------------------------------------------------------
def grafoDorogovtsevMendes(n, dirigido=False):
    """
    Modelo basado en subdividir aristas.

    Se inicia con un triángulo.
    Cada nuevo nodo se conecta a los extremos
    de una arista elegida al azar.
    """

    g = Grafo(dirigido)

    # triángulo inicial
    for i in range(3):
        g.agregar_nodo(i)

    g.agregar_arista(0, 1)
    g.agregar_arista(1, 2)
    g.agregar_arista(2, 0)

    # agregar nodos
    for nuevo in range(3, n):

        g.agregar_nodo(nuevo)

        arista = random.choice(g.aristas)

        u = arista.origen.id
        v = arista.destino.id

        g.agregar_arista(nuevo, u)
        g.agregar_arista(nuevo, v)

    return g

In [ ]:
g1 = grafoMalla(10, 10)
g2 = grafoErdosRenyi(50, 100)
g3 = grafoGilbert(50, 0.05)
g4 = grafoGeografico(50, 0.2)
g5 = grafoBarabasiAlbert(50, 3)
g6 = grafoDorogovtsevMendes(50)

g1.guardar_graphviz("malla.dot")
g2.guardar_graphviz("erdos.dot")
g3.guardar_graphviz("gilbert.dot")
g4.guardar_graphviz("geo.dot")
g5.guardar_graphviz("barabasi.dot")
g6.guardar_graphviz("dm.dot")